In [376]:
# !pip install torchinfo 

In [361]:
import importlib
import src

importlib.reload(src.data_utils)
importlib.reload(src.next_token_dataset)
importlib.reload(src.lstm_model)
importlib.reload(src.lstm_train)

from src.data_utils import text_clean
from src.next_token_dataset import next_token_dataset
from src.lstm_model import next_token_model
from src.lstm_train import train_model

In [409]:
import pandas as pd
import yaml
import evaluate
import torch
from torchinfo import summary
import torch.nn as nn
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datetime import datetime

rouge = evaluate.load("rouge")

## Config

In [ ]:
batch_size = 64
num_epochs = 3
min_len = 16
max_len = 32
tokenizer_model_name = "bert-base-uncased"
embedding_size = 128
hidden_dim = 256

config_path = "main_config"

# with open(f"configs/{config_path}.yaml", "r", encoding="utf-8") as file:
#     config = yaml.safe_load(file)
#     globals().update(config)


In [396]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Data preprocessing

In [397]:
texts = pd.read_csv("data/tweets.txt", engine='python', sep="±±±", header=None, names=["text"])
texts.head()

,text
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,is upset that he can't update his Facebook by ...
2,@Kenichan I dived many times for the ball. Man...
3,my whole body feels itchy and like its on fire
4,"@nationwideclass no, it's not behaving at all...."


In [398]:
texts["text"] = texts["text"].apply(text_clean)
texts.head()

,text
0,awww that s a bummer you shoulda got david car...
1,is upset that he can t update his facebook by ...
2,i dived many times for the ball managed to sav...
3,my whole body feels itchy and like its on fire
4,no it s not behaving at all i m mad why am i h...


In [399]:
train_texts, val_test_texts = train_test_split(texts, test_size=0.2, random_state=42)
val_texts, test_texts = train_test_split(val_test_texts, test_size=0.5, random_state=42)

train_texts = train_texts.reset_index(drop=True)
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)

train_texts.to_csv("data/train.csv", index=False)
val_texts.to_csv("data/val.csv", index=False)
test_texts.to_csv("data/test.csv", index=False)

print(f"Train sample size: {len(train_texts)}")
print(f"Validate sample size: {len(val_texts)}")
print(f"Test sample size: {len(test_texts)}")

Train sample size: 1280398
Validate sample size: 160050
Test sample size: 160050


## Loading from csv

In [402]:
train_dataset, _ = next_token_dataset(tokenizer, min_len=min_len, max_len=max_len).read_csv("data/train.csv", nrows=100000)
val_dataset, val_texts = next_token_dataset(tokenizer, min_len=min_len, max_len=max_len).read_csv("data/val.csv", nrows=10000)

train_loader = train_dataset.get_data_loader(batch_size=batch_size, shuffle=True)
val_loader = val_dataset.get_data_loader(batch_size=batch_size, shuffle=False)


print(f"Train dataset size: {len(train_dataset)}")
print(f"Validate dataset size: {len(val_dataset)}")
# # print(f"Test dataset size: {len(test_dataset)}")

100%|██████████| 9985/9985 [00:00<00:00, 297996.47it/s]
10it [00:04,  2.45it/s]
100%|██████████| 9976/9976 [00:00<00:00, 234552.99it/s]
1it [00:00,  4.68it/s]

Train dataset size: 374643
Validate dataset size: 36980


## Model training

In [403]:
model = next_token_model(
    tokenizer=tokenizer,
    max_len=max_len,
    emb_dim=embedding_size,
    hidden_dim=hidden_dim,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Модель перенесена на девайс: {device}")
if torch.cuda.is_available():
  print(f"Девайс: {torch.cuda.get_device_name(0)}")

print("\nСводка по модели")
print(summary(model))

Модель перенесена на девайс: cpu

Сводка по модели
Layer (type:depth-idx)                   Param #
next_token_model                         --
├─Embedding: 1-1                         3,906,816
├─LSTM: 1-2                              395,264
├─LayerNorm: 1-3                         512
├─Dropout: 1-4                           --
├─Linear: 1-5                            7,844,154
Total params: 12,146,746
Trainable params: 12,146,746
Non-trainable params: 0


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')

train_model(
    num_epochs = num_epochs,
    train_loader = train_loader,
    val_texts = val_texts,
    val_loader = val_loader,
    model = model,
    criterion = criterion,
    optimizer = optimizer,
    scheduler = scheduler,
)

  0%|          | 0/5854 [00:00<?, ?it/s]

/Users/emc2/Проекты/личное/Практикум/Deep learning Engineer/practicum_dle/Sprint 2/Project/src/next_token_dataset.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item[0]) for item in batch]
  3%|▎         | 202/5854 [00:33<15:47,  5.96it/s]


KeyboardInterrupt: 

In [414]:
model_path = f"models/{config_path}_{int(datetime.now().timestamp())}__state_dict"
torch.save(model.state_dict(), model_path)
print(f"Модель сохранена в: {model_path}")

Модель сохранена в: models/main_config_1773862230__state_dict


In [371]:
tweet_idx = 0
tweet = val_texts["text"][tweet_idx]
predict = model.complete_tweet(val_texts["text"][tweet_idx])[0]

tweet_tokenized = tokenizer.encode(tweet)
length = int(len(tweet_tokenized) * 3 / 4)
tokens_to_generate = int(len(tweet_tokenized) * 1 / 4)
subtweet = tokenizer.decode(tweet_tokenized[:length], skip_special_tokens=True)

rouges = model.compute_rouges(val_texts.iloc[tweet_idx:tweet_idx+1])

print(f"Tweet:    {tweet}")
print(f"Subtweet: {subtweet}")
print(f"Predict:  {predict}")
print(f'Rouges:   {rouges}')


Tweet:    nice hair cut dude why were your students leaving in the middle of class 1st period
Subtweet: nice hair cut dude why were your students leaving in the middle
Predict:  nice hair cut dude why were your students leaving in the middle
Rouges:   (np.float64(0.0), np.float64(0.0))


In [374]:
for tweet_idx in tqdm(range(100)):
    tweet = val_texts["text"][tweet_idx]
    predict = model.complete_tweet(val_texts["text"][tweet_idx])[0]

    tweet_tokenized = tokenizer.encode(tweet)
    length = int(len(tweet_tokenized) * 3 / 4)
    tokens_to_generate = int(len(tweet_tokenized) * 1 / 4)
    subtweet = tokenizer.decode(tweet_tokenized[:length], skip_special_tokens=True)

    rouges = model.compute_rouges(val_texts.iloc[tweet_idx:tweet_idx+1])
    if rouges[0] >= 0.7:
        print(f"Tweet:    {tweet}")
        print(f"Subtweet: {subtweet}")
        print(f"Predict:  {predict}")
        print(f'Rouges:   {rouges}')
        break


100%|██████████| 100/100 [00:07<00:00, 14.21it/s]


In [413]:
model_path = f"{config_path}_{int(datetime.now().timestamp())}__state_dict"
model_path


'main_config_1773862174__state_dict'